In [2]:
import torch
import torch.nn as nn
from ultralytics import YOLO

In [3]:
PATH_PT = "../../../models/yolov8n.pt"
PATH_RELU_PT = "../../../models/yolov8n_relu_init.pt"
PATH_DATA = "/home/helbuk/Documents/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/data.yaml"

In [20]:
from pathlib import Path
import yaml

print("PATH_DATA =", PATH_DATA)
print("resolved =", Path(PATH_DATA).resolve())
print("exists =", Path(PATH_DATA).exists())

with open(Path(PATH_DATA).resolve(), "r") as f:
    print(f.read())

PATH_DATA = ../../../datasets/coco/data.yaml
resolved = /home/helbuk/Documents/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/data.yaml
exists = True
path: .

train: train/images
val: valid/images

task: detect

# Classes
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: 

In [33]:
def replace_silu_with_relu(module: nn.Module, inplace: bool = True) -> nn.Module:
    for name, child in list(module.named_children()):
        if isinstance(child, nn.SiLU):
            setattr(module, name, nn.ReLU(inplace=inplace))
        else:
            replace_silu_with_relu(child, inplace=inplace)
    return module

model = YOLO(PATH_PT)
replace_silu_with_relu(model.model)

num_silu = sum(1 for m in model.model.modules() if isinstance(m, nn.SiLU))
num_relu = sum(1 for m in model.model.modules() if isinstance(m, nn.ReLU))
print("remaining SiLU:", num_silu)
print("ReLU count:", num_relu)

ckpt_path = PATH_RELU_PT
torch.save({"model": model.model}, ckpt_path)

remaining SiLU: 0
ReLU count: 57


In [4]:
baseline = YOLO(PATH_PT)
relu_model = YOLO(PATH_RELU_PT)

print("Baseline:")
baseline.val(data=PATH_DATA)

print("ReLU fine-tuned:")
relu_model.val(data=PATH_DATA)

Baseline:
Ultralytics 8.4.6 🚀 Python-3.11.14 torch-2.9.1 CPU (Apple M3 Pro)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs


FileNotFoundError: '/home/helbuk/Documents/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/data.yaml' does not exist

In [5]:
relu_model.model

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): ReLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): ReLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
    

In [6]:
baseline.model

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1))
        (act): SiLU(inplace=True)
      )
      (m): ModuleList(
        (0): Bottleneck(
          (cv1): Conv(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (act): SiLU(inplace=True)
          )
        )
      )
    )
    (3): Conv(
      (conv): Conv2d(32

In [7]:
relu_model.export(
    format="onnx",
    opset=17,
    imgsz=640,
    dynamic=False,
    simplify=True,
    nms=False,       
)

Ultralytics 8.4.6 🚀 Python-3.11.14 torch-2.9.1 CPU (Apple M3 Pro)
YOLOv8n summary (fused): 128 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from '../../../models/yolov8n_relu_init.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (12.3 MB)

ONNX: starting export with onnx 1.20.1 opset 17...
ONNX: slimming with onnxslim 0.1.87...
ONNX: export success ✅ 1.5s, saved as '../../../models/yolov8n_relu_init.onnx' (12.2 MB)

Export complete (1.7s)
Results saved to /Users/helbuk/Documents/NTNU_Thesis_2026/Compiler-Aware_Model_Optimization/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/models
Predict:         yolo predict task=detect model=../../../models/yolov8n_relu_init.onnx imgsz=640 
Validate:        yolo val task=detect model=../../../models/yolov8n_relu_init.onnx imgsz=640 data=None  
Visualize:       https://netron.app


'../../../models/yolov8n_relu_init.onnx'